### Aula 05 — Partições e distribuição do trabalho

O objetivo é acompanhar como a mesma leitura de clientes passa de **1 para 4 e depois para 2 partições**, relacionando os números aos operadores do plano físico. A sequência compara a redistribuição de `repartition(4)` com a redução de `coalesce(2)`.

Uma **Spark Partition** é uma unidade lógica de dados usada para dividir o processamento. Em cada stage, normalmente há uma task por partição a processar; os **executors** executam essas tasks com os recursos disponíveis. Quatro partições não representam quatro executors e não garantem quatro tasks simultâneas: a concorrência depende dos núcleos e slots disponíveis. O driver coordena esse trabalho.

As próximas células importam `SparkSession` e obtêm a sessão da aplicação. `getOrCreate()` pode reutilizar uma sessão existente; o nome da aplicação não define o tamanho do cluster.

**Como ler as evidências:** os resultados abaixo são saídas já salvas no notebook, não medições novas. Para estudar em outro ambiente, o CSV precisa estar disponível no caminho indicado pelo código. A quantidade inicial de partições depende dos arquivos, do tamanho e das configurações de leitura.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_05_particoes")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 16:38:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### 1. Ingestão do CSV de clientes

`spark.read.csv()` cria o DataFrame `df` a partir de `05_particoes/clientes.csv`. `header=True` usa o cabeçalho como nome das colunas; `inferSchema=True` solicita inferência dos tipos, que pode demandar leitura dos dados. O CSV não armazena um schema tipado como parte do formato.

No plano salvo, aparecem tipos como `cliente_id: int`, `renda_mensal: double`, `ativo: boolean` e `data_cadastro: date`. Conhecer os tipos evita comparações e cálculos inconsistentes nas próximas etapas de um pipeline de Engenharia de Dados.

In [3]:
df = spark.read.csv("/workspace/datasets/05_particoes/clientes.csv", header= True, inferSchema= True)

#### 2. Particionamento inicial e inspeção da leitura

`df.rdd.getNumPartitions()` consulta o particionamento do DataFrame pela representação RDD. **Resultado salvo:** `1`, repetido mais adiante por `print(...)`. Esse número descreve esta leitura; não implica que todo CSV tenha uma partição.

A consulta de partições não equivale a contar todas as linhas. A conversão para RDD e a consulta podem exigir planejamento e, com execução adaptativa, materialização de etapas; por isso, não se deve assumir que essa inspeção seja sempre livre de trabalho Spark. `show()` é uma action e calcula o necessário para exibir a amostra de clientes, limitada aqui a 20 linhas. A amostra não informa o tamanho total do DataFrame.

In [4]:
df.rdd.getNumPartitions()

1

##### 2.1. O plano antes de redistribuir

`explain(True)` apresenta o plano lógico recebido, sua análise de nomes e tipos, o plano otimizado pelo Catalyst e a estratégia física. Ele permite inspecionar a consulta sem pedir seu resultado completo.

No **Physical Plan salvo**, há apenas `FileScan csv`: a leitura da fonte com as colunas listadas em `ReadSchema`. `DataFilters`, `PushedFilters` e `PartitionFilters` estão vazios porque este DataFrame não possui filtro nem leitura seletiva por diretórios particionados. Não há `Exchange` nesta leitura.

Esse ponto de partida ajuda a distinguir o custo da fonte do custo acrescentado pelas transformações seguintes. Uma transformação **narrow**, como `select`, `filter` ou `withColumn`, trabalha com dependências locais e não exige redistribuir os registros por chave. Uma transformação **wide**, como `repartition`, cria dependências entre partições. Essa classificação depende das dependências da operação, e não de existir uma ou muitas partições inicialmente.

In [5]:
df.explain(True)

== Parsed Logical Plan ==
Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/05_particoes/clientes.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,n

In [6]:
print(df.rdd.getNumPartitions())

1


In [7]:
df.show()

+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|      nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|         1| Cliente 1|   32|Feira de Santana|    BA|   Engenheiro|    12925.67|   M|false|   2025-02-22|
|         2| Cliente 2|   32|Feira de Santana|    BA|   Engenheiro|     4852.23|   M|false|   2024-01-16|
|         3| Cliente 3|   29|Feira de Santana|    BA|   Engenheiro|      8919.6|   F|false|   2023-02-08|
|         4| Cliente 4|   64|        Salvador|    BA|     Analista|     4253.53|   F|false|   2023-01-03|
|         5| Cliente 5|   60|        Salvador|    BA|   Engenheiro|     2710.92|   M|false|   2023-09-15|
|         6| Cliente 6|   30|        Salvador|    BA|   Engenheiro|    10282.08|   F|false|   2023-01-29|
|         7| Cliente 7|   46|Feira de Santana|

#### 3. `repartition(4)`: redistribuir os registros

`df.repartition(4)` constrói `df_repartition` com uma redistribuição para quatro partições. Como nenhuma coluna de particionamento foi fornecida, o plano salvo usa `RoundRobinPartitioning(4)`, distribuindo linhas entre as partições de saída sem agrupá-las por uma chave de negócio.

**Comportamento esperado:** uma fronteira de shuffle, com custo de serialização, transferência e possível uso de disco. Em volumes maiores, redistribuir pode ampliar o paralelismo disponível, mas o ganho depende do trabalho e dos recursos; este notebook não mede duração nem comprova aceleração.

**Evidências salvas:** `getNumPartitions()` retorna `4`, e a amostra de `show()` passa a começar com outros clientes. Redistribuir não ordena os dados: a mudança na ordem de exibição não representa perda de registros, e a amostra isolada também não prova igualdade de contagens.

In [8]:
df_repartition = df.repartition(4)

In [9]:
df_repartition.rdd.getNumPartitions()

4

In [10]:
df_repartition.show()

+----------+-----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|       nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+-----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|       225|Cliente 225|   21|       São Paulo|    SP|Desenvolvedor|    12893.39|   F|false|   2023-03-27|
|       201|Cliente 201|   67|        Salvador|    BA|Desenvolvedor|     1905.65|   F|false|   2023-10-17|
|       109|Cliente 109|   49|       São Paulo|    SP|    Professor|    12791.96|   M|false|   2023-07-30|
|       206|Cliente 206|   65|       São Paulo|    SP|Desenvolvedor|     14243.3|   F|false|   2023-10-24|
|        18| Cliente 18|   58|       São Paulo|    SP|    Professor|     5460.96|   M|false|   2025-05-08|
|        99| Cliente 99|   48|Feira de Santana|    BA|   Engenheiro|     7330.62|   F| true|   2025-02-20|
|       300|Cliente 300|   44|       

##### 3.1. `Exchange` e o plano adaptativo observado

Leia o plano físico de baixo para cima: `FileScan csv` fornece as linhas; `Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM` estabelece sua nova distribuição. O **shuffle** é a movimentação necessária para essa distribuição; o `Exchange` não soma nem filtra os registros. Essa fronteira separa etapas de execução e exige trabalho para produzir e consumir as partições redistribuídas.

O output mostra **`AdaptiveSparkPlan isFinalPlan=true`**, com `Final Plan` e `Initial Plan`. O plano final contém `ShuffleQueryStage 0` em torno do mesmo Exchange. Esse wrapper registra o estágio de shuffle materializado no contexto da execução adaptativa (AQE); não é uma quinta partição nem um novo `repartition`.

AQE pode adaptar decisões com estatísticas obtidas em execução. Neste resultado específico, a distribuição explícita de quatro partições permanece visível. A presença de AQE não é sinônimo de shuffle: o operador que evidencia a redistribuição aqui é o `Exchange`.

In [11]:
df_repartition.explain(True)

== Parsed Logical Plan ==
Repartition 4, true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Repartition 4, true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Repartition 4, true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 0
   +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=40]
      +- FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25

#### 4. `coalesce(2)`: reduzir o paralelismo

`df_repartition.coalesce(2)` parte das quatro partições anteriores e reduz a saída para duas. O `coalesce` sem shuffle agrupa partições existentes, sem fazer outra redistribuição completa de todas as linhas. **Resultado salvo:** `2` em `getNumPartitions()`.

Essa redução pode ser útil para ajustar a quantidade de trabalho ou de arquivos de uma escrita futura. Entretanto, menos partições também significa menos tasks na etapa seguinte, e partições grandes ou desiguais podem criar gargalos. `coalesce` não garante equilíbrio e não é uma forma de aumentar o número de partições.

Não há escrita de arquivos neste notebook: a comparação observada se limita ao particionamento e aos planos, sem medir tamanhos de arquivos ou tempo de execução.

In [12]:
df_coalesce = df_repartition.coalesce(2)

In [13]:
df_coalesce.rdd.getNumPartitions()

2

##### 4.1. Por que ainda existe `Exchange` após `coalesce`?

O plano salvo contém `Coalesce 2` sobre `ShuffleQueryStage 0` e `Exchange RoundRobinPartitioning(4)`. O Exchange pertence ao **`repartition(4)` anterior**, preservado no lineage; não foi criado pelo `coalesce(2)`.

No plano lógico, `Repartition 2, false` indica a redução sem shuffle, enquanto `Repartition 4, true` identifica a redistribuição anterior. No plano físico final, `Coalesce 2` consome esse resultado e reduz suas partições. Os blocos `Initial Plan` e `Final Plan` são duas representações do planejamento, não duas execuções integrais obrigatórias do pipeline.

A distinção é relevante ao diagnosticar um pipeline: localizar um Exchange em algum ponto do plano não basta para atribuir seu custo à última função chamada.

In [14]:
df_coalesce.explain(True)

== Parsed Logical Plan ==
Repartition 2, false
+- Repartition 4, true
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Repartition 2, false
+- Repartition 4, true
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Repartition 2, false
+- Repartition 4, true
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   Coalesce 2
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=85]
         +- FileSc

#### 5. Encerramento da sessão

`spark.stop()` encerra os recursos associados à sessão/contexto. Esta célula não possui contador de execução nem saída salva; sua presença não comprova que o encerramento tenha sido executado nesta cópia do notebook.

In [ ]:
spark.stop()

#### Resumo dos aprendizados

| Etapa | Partições registradas | Evidência no plano salvo |
|---|---:|---|
| CSV original | 1 | `FileScan csv`, sem Exchange |
| `repartition(4)` | 4 | `Exchange RoundRobinPartitioning(4)` |
| `coalesce(2)` após o repartition | 2 | `Coalesce 2` sobre o Exchange anterior |

Partições dividem os dados; tasks processam esse trabalho; executors fornecem recursos para executá-lo. Aumentar o número de partições pode disponibilizar mais paralelismo, mas acrescenta custo e não implica mais executors. `repartition` redistribui; `coalesce` reduz sem introduzir aqui um novo shuffle completo. Os resultados `1 → 4 → 2` e os planos são as evidências existentes; não há medição de desempenho ou escrita neste experimento.

Para aprofundar, consulte [o capítulo de partições](../docs/05_particoes.md). O [Desafio 05](../desafios/desafio_05_particoes.ipynb) aplica esses conceitos a uma sequência com seleção, filtro e colunas derivadas.